In [4]:
from sqlitesearch import TextSearchIndex

sqlite_index = TextSearchIndex(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"],
    db_path="faq.db"
)

In [5]:
sqlite_index.count()

79

In [6]:
results = sqlite_index.search("Can I still join the course after it started?", num_results=5)
[doc["question"] for doc in results]

['I just discovered the course. Can I still join?',
 'I missed the first homework - can I still get a certificate?',
 'Do we submit 2 projects, what does attempt 1 and 2 mean?',
 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
 'I am using Azure OpenAI and I am still getting an error of Error code: 400 - {\'error\': {\'message\': "Missing required parameter: \'tools[0].function\'.", \'type\': \'invalid_request_error\', \'param\': \'tools[0].function\', \'code\': \'missing_required_parameter\'}}?']

In [7]:
from rag_helper import RAGBase
from openai import OpenAI

openai_client = OpenAI()

assistant = RAGBase(
    index=sqlite_index,
    llm_client=openai_client,
)

In [9]:
answer = assistant.rag("Can I still join the course after it started?")
print(answer)

Yes, you can still join the course after it started. If you want a certificate, you need to submit your project while submissions are still open.


In [10]:
sqlite_index.close()

## 12-rag-revision

In [11]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

from rag_helper import RAGBase
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=instructions,
)

In [12]:
assistant.rag("How do I run Ollama locally?")

'To run Ollama locally:\n\n1. Install Ollama from: https://ollama.com/download  \n   - macOS: install the `.pkg`\n   - Windows: install the `.msi`\n   - Linux: run:\n   ```bash\n   curl -fsSL https://ollama.com/install.sh | sh\n   ```\n\n2. Open a terminal and run:\n```bash\nollama run llama3\n```\n\nThis will download the LLaMA 3 model, start it locally, and open a chat-like interface.\n\n3. To test that the local Ollama server is running, you can run:\n```bash\ncurl http://localhost:11434\n```\n\nIf you’re in the notebook homework environment and get a connection refused error, restart the Ollama server with:\n```bash\n!nohup ollama serve > nohup.out 2>&1 &\n```'

In [13]:
assistant.rag("How do I run Olama locally?")

'I don’t have any information in the provided context about running **Olama locally**.\n\nThe only related setup info in the FAQ is about running the MCP Inspector:\n\n```bash\nnpx @modelcontextprotocol/inspector\n```\n\nIf you want, I can help with the Olama/Local LLM setup based on other instructions you provide.'

## 13-function-calling

In [14]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [15]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [17]:
messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output

[ResponseFunctionToolCall(arguments='{"query":"Can I join the course after it has started? discovered the course late enrollment"}', call_id='call_MyOlkdB2Nl1CSzDRGZLToLq0', name='search', type='function_call', id='fc_05a8da290975d9ba006a23e7b702c8819a853ecc14483b9674', namespace=None, status='completed')]

In [18]:
import json

call = response.output[0]
args = json.loads(call.arguments)

results = search(**args)
result_json = json.dumps(results, indent=2)

In [19]:
messages.extend(response.output)

messages.append({
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json,
})

In [20]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output_text

'Yes — you can still join the course.\n\nIf you want a certificate, make sure you submit your project while submissions are still open.'

In [21]:
usage = response.usage
usage.input_tokens, usage.output_tokens

(656, 31)

In [23]:
def calculate_gpt54mini_price(input_tokens, output_tokens):
    INPUT_PRICE_PER_MILLION = 0.15
    OUTPUT_PRICE_PER_MILLION = 0.60

    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION
    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost,
    }

result = calculate_gpt54mini_price(656, 31)
print("Total cost: $", round(result["total_cost"], 8))

Total cost: $ 0.000117


## 14-agentic-loop

In [24]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

In [25]:
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == "search":
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }

In [26]:
question = "I just discovered the course. Can I join it?"

messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question},
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

messages.extend(response.output)
has_function_calls = False

for item in response.output:
    if item.type == "function_call":
        print("function_call:", item.name, item.arguments)
        call_output = make_call(item)
        messages.append(call_output)
        has_function_calls = True

    elif item.type == "message":
        print("ASSISTANT:")
        print(item.content[0].text)

function_call: search {"query":"join the course enroll discovered course can I join"}
function_call: search {"query":"course enrollment joining late can I join discovered course"}
function_call: search {"query":"course access late enrollment registration FAQ"}


In [27]:
it = 1

while True:
    print(f"iteration #{it}...")
    has_function_calls = False

    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=messages,
        tools=[search_tool],
    )

    messages.extend(response.output)

    for item in response.output:
        if item.type == "function_call":
            print("function_call:", item.name, item.arguments)
            call_output = make_call(item)
            messages.append(call_output)
            has_function_calls = True

        elif item.type == "message":
            print("ASSISTANT:")
            print(item.content[0].text)

    it = it + 1
    if has_function_calls == False:
        break

iteration #1...
ASSISTANT:
Yes — you can still join the course.

If you want a certificate, you’ll need to submit your project while submissions are still open.

Also, if you’ve registered already, you don’t need to wait for a confirmation email; you can just start learning and submit homework while the form is open.

Would you like help with anything else about the course?


In [28]:
def agent_loop(instructions, question, model="gpt-5.4-mini") -> str:
    messages = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question}
    ]

    it = 1

    while True:
        print(f"iteration #{it}...")
        has_function_calls = False

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool]
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == "function_call":
                print("function_call:", item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == "message":
                print("ASSISTANT:")
                last_answer = item.content[0].text
                print(item.content[0].text)

        it = it + 1
        if has_function_calls == False:
            break

    return last_answer

In [29]:
agent_loop(instructions, "How do I run Olama locally?")

iteration #1...
function_call: search {"query":"Olama locally run install local model course FAQ Ollama"}
function_call: search {"query":"run Ollama locally command install ollama FAQ"}
iteration #2...
ASSISTANT:
To run Ollama locally:

1. **Install Ollama**
   - **macOS**: download and install the `.pkg` from https://ollama.com/download
   - **Windows**: download and install the `.msi`
   - **Linux**:
     ```bash
     curl -fsSL https://ollama.com/install.sh | sh
     ```

2. **Start a model locally**
   ```bash
   ollama run llama3
   ```
   This will download the model, start it locally, and open a chat-style prompt.

3. **Check that the local server is running**
   ```bash
   curl http://localhost:11434
   ```

4. **Use it from Python**
   ```bash
   pip install ollama
   ```

   ```python
   import ollama

   response = ollama.chat(
       model='llama3',
       messages=[{"role": "user", "content": "Hello!"}]
   )

   print(response['message']['content'])
   ```

If you run into

'To run Ollama locally:\n\n1. **Install Ollama**\n   - **macOS**: download and install the `.pkg` from https://ollama.com/download\n   - **Windows**: download and install the `.msi`\n   - **Linux**:\n     ```bash\n     curl -fsSL https://ollama.com/install.sh | sh\n     ```\n\n2. **Start a model locally**\n   ```bash\n   ollama run llama3\n   ```\n   This will download the model, start it locally, and open a chat-style prompt.\n\n3. **Check that the local server is running**\n   ```bash\n   curl http://localhost:11434\n   ```\n\n4. **Use it from Python**\n   ```bash\n   pip install ollama\n   ```\n\n   ```python\n   import ollama\n\n   response = ollama.chat(\n       model=\'llama3\',\n       messages=[{"role": "user", "content": "Hello!"}]\n   )\n\n   print(response[\'message\'][\'content\'])\n   ```\n\nIf you run into a connection issue, restarting the server with `ollama serve` can help.\n\nDo you want guidance for a specific OS or for using Ollama inside Python/Jupyter?'

In [30]:
agent_loop(instructions, "I just discovered the course. Can I still join it?")

iteration #1...
function_call: search {"query":"join course late enrollment discovered course can I still join"}
iteration #2...
ASSISTANT:
Yes, you can still join the course. If you want a certificate, though, you’ll need to submit your project while submissions are still being accepted.

If you’d like, I can also help explain how certificates, homework, or self-paced participation work. Any other areas you want to explore?


'Yes, you can still join the course. If you want a certificate, though, you’ll need to submit your project while submissions are still being accepted.\n\nIf you’d like, I can also help explain how certificates, homework, or self-paced participation work. Any other areas you want to explore?'

In [31]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "I just discovered the course. Can I join it?")

iteration #1...
function_call: search {"query":"join course discovered course can I join enrollment late join FAQ"}
iteration #2...
function_call: search {"query":"certificate submit project while accepting submissions peer review live cohort self-paced mode FAQ"}
iteration #3...
ASSISTANT:
Yes — you can still join the course.

If you want a certificate, make sure you submit your project while submissions are still open, since certificates require participating in the live cohort and peer-review process.

If you want, I can also explain how the certificate process works or whether you can take it in self-paced mode.


'Yes — you can still join the course.\n\nIf you want a certificate, make sure you submit your project while submissions are still open, since certificates require participating in the live cohort and peer-review process.\n\nIf you want, I can also explain how the certificate process works or whether you can take it in self-paced mode.'

In [32]:
agent_loop(instructions, "what's queen gambit?")

iteration #1...
function_call: search {"query":"queen gambit queen's gambit opening chess definition"}
iteration #2...
function_call: search {"query":"queen's gambit chess opening king pawn queen gambit meaning"}
iteration #3...
ASSISTANT:
A **Queen’s Gambit** is a chess opening:

- It starts with **1. d4 d5 2. c4**
- White offers the **c-pawn** as a “gambit” to try to gain control of the center
- It’s one of the most famous and well-studied openings in chess

A simple idea behind it:
- White challenges Black’s center pawn on **d5**
- If Black takes the c-pawn, White usually gets a stronger center or better piece activity

There are two main types:
- **Queen’s Gambit Accepted**: Black takes the c-pawn
- **Queen’s Gambit Declined**: Black does not take it

If you want, I can also explain **why it’s called a gambit** or show the **first few moves and plans** for both sides.


'A **Queen’s Gambit** is a chess opening:\n\n- It starts with **1. d4 d5 2. c4**\n- White offers the **c-pawn** as a “gambit” to try to gain control of the center\n- It’s one of the most famous and well-studied openings in chess\n\nA simple idea behind it:\n- White challenges Black’s center pawn on **d5**\n- If Black takes the c-pawn, White usually gets a stronger center or better piece activity\n\nThere are two main types:\n- **Queen’s Gambit Accepted**: Black takes the c-pawn\n- **Queen’s Gambit Declined**: Black does not take it\n\nIf you want, I can also explain **why it’s called a gambit** or show the **first few moves and plans** for both sides.'

In [33]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "what's queen gambit?")

iteration #1...
function_call: search {"query":"queen gambit"}
iteration #2...
function_call: search {"query":"queen's gambit chess opening course FAQ"}
iteration #3...
ASSISTANT:
I couldn’t find anything in the course FAQ about “queen gambit,” so it looks like this is likely off-topic for the course.

If you meant something course-related, feel free to rephrase it, and I’ll check the FAQ again. Is there another area you want to explore?


'I couldn’t find anything in the course FAQ about “queen gambit,” so it looks like this is likely off-topic for the course.\n\nIf you meant something course-related, feel free to rephrase it, and I’ll check the FAQ again. Is there another area you want to explore?'

## 15-frameworks

The handwritten agent loop from the previous lesson is educational but repetitive. Every time you build a new agent, you'd write the same while-loop, the same function-call handling, the same message management.

ToyAIKit wraps this pattern so you can focus on tools, prompts, and behavior. We built it together in a DataTalks.Club workshop a while back. It does the same thing as our handwritten loop with less boilerplate. If you open its runners code, you'll find the same while True loop we wrote by hand.

In [34]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [35]:
agent_tools = Tools()
agent_tools.add_tool(search, search_tool)

The output is the same JSON schema we hand-wrote in the function calling lesson. ToyAIKit generated it from the docstring and the type hint.

Every modern agent framework does this same trick. It reads a typed Python function with a docstring and builds the schema from it. The OpenAI Agents SDK, PydanticAI, LangChain and Google ADK all work this way. You write the tool and the framework figures out how to describe it.



In [36]:
def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={"question": 3.0, "section": 0.5},
        filter_dict={"course": "llm-zoomcamp"}
    )

In [37]:
agent_tools = Tools()
agent_tools.add_tool(search)

In [38]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the FAQ database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

### Create the chat interface and a callback, then build the runner:



In [39]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

In [40]:
result = runner.loop(
    prompt="How do I run Olama locally?",
    callback=callback,
)

-> Response received


-> Response received


-> Response received


We used the typo "Olama" on purpose. The agent searches and gets poor results, then retries with "Ollama". The recovery is the same as the handwritten loop. The notebook output is nicer to watch. Each tool call and message renders inline, so you can look at every search result.

The result is a LoopResult with all_messages (the full conversation), token counts, and cost (computed from token usage).



In [41]:
result.cost

CostInfo(input_cost=Decimal('0.0025515'), output_cost=Decimal('0.001341'), total_cost=Decimal('0.0038925'))

In [43]:
result2 = runner.loop(
    prompt="How do I run a different model?",
    previous_messages=result.all_messages,
    callback=callback,
)

-> Response received


-> Response received


In [44]:
runner.run()

-> Response received


-> Response received


-> Response received


-> Response received


Chat ended.


LoopResult(new_messages=[EasyInputMessage(content="You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches. First perform search, analyze the results \nand then perform more searches. \n\nThe question has to be about the course or its logistics, offtopic questions \nshouldn't be answered. If the search returns nothing, it's likely an off-topic question.\nIf you can't answer the question using FAQ, don't do it yourself. Only use the \nfacts from the FAQ database.\n\nAt the end, ask if there are other areas that the user wants to explore.", role='developer', phase=None, type=None), EasyInputMessage(content='Hello, how do I get certificate?', role='user', phase=None, type=None), ResponseFunctionToolCall(arguments='{"query":"certificate get certificate how do 